In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
#ADC_WORKS_TITLE_VARIANT_GENERATION

# import re
# import snowflake.snowpark.functions as F
# from snowflake.snowpark.functions import col, lit, regexp_replace, when

# # SQL UDFs for core string manipulation functions
# def register_udfs(session):
#     print("Registering JavaScript UDFs for title variant generation...")
    
#     # Register the JavaScript UDFs
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.IS_ARTICLE(word STRING)
#     RETURNS BOOLEAN
#     LANGUAGE JAVASCRIPT
#     AS $$
#         const articles = ['A', 'AN', 'THE'];
#         return articles.includes(WORD.toUpperCase());
#     $$;
#     """).collect()
    
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.REMOVE_ARTICLES(title STRING)
#     RETURNS STRING
#     LANGUAGE JAVASCRIPT
#     AS $$
#         if (!TITLE) return '';
        
#         const articles = ['A', 'AN', 'THE'];
#         const words = TITLE.trim().split(/\s+/);
        
#         // Remove leading article
#         if (words.length > 0 && articles.includes(words[0].toUpperCase())) {
#             words.shift();
#         }
        
#         // Remove trailing article
#         if (words.length > 0 && articles.includes(words[words.length - 1].toUpperCase())) {
#             words.pop();
#         }
        
#         return words.join(' ');
#     $$;
#     """).collect()
    
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.NORMALIZE_WHITESPACE(title STRING)
#     RETURNS STRING
#     LANGUAGE JAVASCRIPT
#     AS $$
#         if (!TITLE) return '';
#         return TITLE.trim().replace(/\s+/g, ' ');
#     $$;
#     """).collect()
    
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.CLEAN_UNMATCHED_BRACKETS(title STRING)
#     RETURNS STRING
#     LANGUAGE JAVASCRIPT
#     AS $$
#         if (!TITLE) return '';
        
#         const stack = [];
#         const bracketPairs = {'(': ')', '[': ']', '{': '}'};
#         const reversePairs = {')': '(', ']': '[', '}': '{'};
#         const skipIndices = new Set();
        
#         // First pass - identify unmatched brackets
#         for (let i = 0; i < TITLE.length; i++) {
#             const char = TITLE[i];
            
#             if (bracketPairs[char]) {  // Opening bracket
#                 stack.push([char, i]);
#             } else if (reversePairs[char]) {  // Closing bracket
#                 if (stack.length && stack[stack.length - 1][0] === reversePairs[char]) {
#                     stack.pop();  // Matched pair
#                 } else {
#                     // Unmatched closing bracket - mark for removal
#                     skipIndices.add(i);
#                 }
#             }
#         }
        
#         // Any remaining opening brackets are unmatched
#         for (const [_, idx] of stack) {
#             skipIndices.add(idx);
#         }
        
#         // Build cleaned title without unmatched brackets
#         let cleanedTitle = "";
#         for (let i = 0; i < TITLE.length; i++) {
#             if (!skipIndices.has(i)) {
#                 cleanedTitle += TITLE[i];
#             }
#         }
        
#         return cleanedTitle.trim().replace(/\s+/g, ' ');
#     $$;
#     """).collect()
    
#     # Complex variant generation function
#     # This is a simplified JavaScript version that returns JSON with variants
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.GENERATE_TITLE_VARIANTS_JS(title STRING)
#     RETURNS VARIANT
#     LANGUAGE JAVASCRIPT
#     AS $$
#         // Handle NULL titles
#         if (TITLE === null || TITLE === undefined) {
#             return [];
#         }
        
#         const originalTitle = TITLE.trim();
        
#         // Helper functions
#         function isArticle(word) {
#             const articles = ['A', 'AN', 'THE'];
#             return articles.includes(word.toUpperCase());
#         }
        
#         function removeArticles(title) {
#             if (!title) return '';
#             const words = title.trim().split(/\s+/);
            
#             if (words.length > 0 && isArticle(words[0])) {
#                 words.shift();
#             }
            
#             if (words.length > 0 && isArticle(words[words.length - 1])) {
#                 words.pop();
#             }
            
#             return words.join(' ');
#         }
        
#         function normalizeWhitespace(title) {
#             if (!title) return '';
#             return title.trim().replace(/\s+/g, ' ');
#         }
        
#         function cleanUnmatchedBrackets(title) {
#             if (!title) return '';
            
#             const stack = [];
#             const bracketPairs = {'(': ')', '[': ']', '{': '}'};
#             const reversePairs = {')': '(', ']': '[', '}': '{'};
#             const skipIndices = new Set();
            
#             // First pass - identify unmatched brackets
#             for (let i = 0; i < title.length; i++) {
#                 const char = title[i];
                
#                 if (bracketPairs[char]) {  // Opening bracket
#                     stack.push([char, i]);
#                 } else if (reversePairs[char]) {  // Closing bracket
#                     if (stack.length && stack[stack.length - 1][0] === reversePairs[char]) {
#                         stack.pop();  // Matched pair
#                     } else {
#                         // Unmatched closing bracket - mark for removal
#                         skipIndices.add(i);
#                     }
#                 }
#             }
            
#             // Any remaining opening brackets are unmatched
#             for (const [_, idx] of stack) {
#                 skipIndices.add(idx);
#             }
            
#             // Build cleaned title without unmatched brackets
#             let cleanedTitle = "";
#             for (let i = 0; i < title.length; i++) {
#                 if (!skipIndices.has(i)) {
#                     cleanedTitle += title[i];
#                 }
#             }
            
#             return normalizeWhitespace(cleanedTitle);
#         }
        
#         // Main variant generation logic
#         const cleanedTitle = cleanUnmatchedBrackets(originalTitle);
#         const variants = new Set();
        
#         // FIXED: Store only the regex pattern source, not the regex object itself
#         const bracketPatternSource = /([\[\(\{])([^\[\]\(\)\{\}]*)([\]\)\}])/g.source;
        
#         // Find all matched brackets in the cleaned title
#         const bracketMatches = [];
#         let match;
#         // FIXED: Always create a new RegExp instance when using the pattern
#         const patternForMatching = new RegExp(bracketPatternSource, 'g');
#         while ((match = patternForMatching.exec(cleanedTitle)) !== null) {
#             bracketMatches.push({
#                 full: match[0],
#                 open: match[1],
#                 content: match[2],
#                 close: match[3],
#                 start: match.index,
#                 end: match.index + match[0].length
#             });
#         }
        
#         // If no brackets, check if original had any
#         if (bracketMatches.length === 0) {
#             if (/[\[\]\(\)\{\}]/.test(originalTitle)) {
#                 const cleanTitle = removeArticles(cleanedTitle);
#                 if (cleanTitle) {
#                     variants.add(cleanTitle);
#                 }
#             }
#             return Array.from(variants).sort();
#         }
        
#         // FIXED: Create a new RegExp instance for each replacement
#         const noBrackets = cleanedTitle.replace(new RegExp(bracketPatternSource, 'g'), '');
#         const normalizedNoBrackets = normalizeWhitespace(noBrackets);
#         const noArticlesNoBrackets = removeArticles(normalizedNoBrackets);
        
#         if (noArticlesNoBrackets && 
#             noArticlesNoBrackets.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#             variants.add(noArticlesNoBrackets);
#         }
        
#         // Group adjacent brackets
#         const bracketGroups = [];
#         if (bracketMatches.length > 0) {
#             let currentGroup = [bracketMatches[0]];
            
#             for (let i = 1; i < bracketMatches.length; i++) {
#                 const prevEnd = bracketMatches[i-1].end;
#                 const currentStart = bracketMatches[i].start;
                
#                 // Check if there's text between brackets
#                 if (cleanedTitle.substring(prevEnd, currentStart).trim()) {
#                     // Non-adjacent brackets, start a new group
#                     bracketGroups.push({
#                         start: currentGroup[0].start,
#                         end: currentGroup[currentGroup.length-1].end,
#                         matches: [...currentGroup]
#                     });
#                     currentGroup = [bracketMatches[i]];
#                 } else {
#                     // Adjacent brackets, add to current group
#                     currentGroup.push(bracketMatches[i]);
#                 }
#             }
            
#             // Add the last group
#             bracketGroups.push({
#                 start: currentGroup[0].start,
#                 end: currentGroup[currentGroup.length-1].end,
#                 matches: [...currentGroup]
#             });
#         }
        
#         // Split the title into segments: text segments and bracket groups
#         const segments = [];
#         let lastEnd = 0;
        
#         for (const group of bracketGroups) {
#             // Add text before this group
#             if (group.start > lastEnd) {
#                 segments.push({
#                     type: 'text',
#                     content: cleanedTitle.substring(lastEnd, group.start)
#                 });
#             }
            
#             // Add this bracket group
#             segments.push({
#                 type: 'group',
#                 content: group.matches
#             });
            
#             // Update lastEnd
#             lastEnd = group.end;
#         }
        
#         // Add any remaining text after the last bracket group
#         if (lastEnd < cleanedTitle.length) {
#             segments.push({
#                 type: 'text',
#                 content: cleanedTitle.substring(lastEnd)
#             });
#         }
        
#         // Variant 1: with first bracket of first group only
#         const firstVariantParts = [];
        
#         for (const segment of segments) {
#             if (segment.type === 'text') {
#                 firstVariantParts.push(segment.content);
#             } else if (segment.type === 'group') {
#                 // Add only first bracket content from first group
#                 firstVariantParts.push(segment.content[0].content);
#             }
#         }
        
#         const firstVariant = removeArticles(normalizeWhitespace(firstVariantParts.join(' ')));
        
#         if (firstVariant && 
#             !variants.has(firstVariant) && 
#             firstVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#             variants.add(firstVariant);
#         }
        
#         // Variant 2: If multiple groups, create variant with first bracket of first group
#         // and first bracket of last group
#         if (bracketGroups.length > 1) {
#             const secondVariantParts = [];
#             let firstGroupUsed = false;
#             let lastGroupUsed = false;
            
#             for (const segment of segments) {
#                 if (segment.type === 'text') {
#                     secondVariantParts.push(segment.content);
#                 } else if (segment.type === 'group') {
#                     // Check if this is first group
#                     if (!firstGroupUsed && segment.content[0].start === bracketGroups[0].matches[0].start) {
#                         secondVariantParts.push(segment.content[0].content);
#                         firstGroupUsed = true;
#                     } 
#                     // Check if this is last group
#                     else if (!lastGroupUsed && 
#                             segment.content[0].start === bracketGroups[bracketGroups.length-1].matches[0].start &&
#                             bracketGroups[0].start !== bracketGroups[bracketGroups.length-1].start) {
#                         secondVariantParts.push(segment.content[0].content);
#                         lastGroupUsed = true;
#                     }
#                 }
#             }
            
#             const secondVariant = removeArticles(normalizeWhitespace(secondVariantParts.join(' ')));
            
#             if (secondVariant && 
#                 !variants.has(secondVariant) && 
#                 secondVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                 variants.add(secondVariant);
#             }
#         }
        
#         // Variant 3: If multiple groups, create variant with just text and first bracket of last group
#         if (bracketGroups.length > 1) {
#             const thirdVariantParts = [];
#             let lastGroupUsed = false;
            
#             for (const segment of segments) {
#                 if (segment.type === 'text') {
#                     thirdVariantParts.push(segment.content);
#                 } else if (segment.type === 'group' && !lastGroupUsed && 
#                         segment.content[0].start === bracketGroups[bracketGroups.length-1].matches[0].start) {
#                     thirdVariantParts.push(segment.content[0].content);
#                     lastGroupUsed = true;
#                 }
#             }
            
#             const thirdVariant = removeArticles(normalizeWhitespace(thirdVariantParts.join(' ')));
            
#             if (thirdVariant && 
#                 !variants.has(thirdVariant) && 
#                 thirdVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                 variants.add(thirdVariant);
#             }
#         }
        
#         // Variant 4: Handle titles that start with a bracket
#         if (bracketMatches.length > 0 && bracketMatches[0].start === 0) {
#             // Get just the content of the first bracket
#             const bracketContent = normalizeWhitespace(bracketMatches[0].content);
#             let restOfTitle = cleanedTitle.substring(bracketMatches[0].end);
            
#             // FIXED: Use a new RegExp instance for replacement
#             restOfTitle = restOfTitle.replace(new RegExp(bracketPatternSource, 'g'), '');
#             restOfTitle = normalizeWhitespace(restOfTitle);
            
#             // Create variant with just bracket content + rest of title
#             const combined = removeArticles(normalizeWhitespace(`${bracketContent} ${restOfTitle}`));
            
#             if (combined && 
#                 !variants.has(combined) && 
#                 combined.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                 variants.add(combined);
#             }
            
#             // Also create a variant with just the bracket content if it's followed by more brackets
#             if (bracketMatches.length > 1 && bracketMatches[1].start === bracketMatches[0].end) {
#                 const justContent = removeArticles(bracketContent);
                
#                 if (justContent && 
#                     !variants.has(justContent) && 
#                     justContent.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                     variants.add(justContent);
#                 }
#             }
#         }
        
#         // Complex case handling for titles like "(NEW WORLD) (HI) HELLO (SMILE TIME) WORLD (TIME) (XXXX)"
#         if (bracketGroups.length >= 2) {
#             // Find brackets that aren't in any group (middle brackets)
#             const middleBrackets = [];
            
#             for (const match of bracketMatches) {
#                 let isInGroup = false;
                
#                 for (const group of bracketGroups) {
#                     for (const groupMatch of group.matches) {
#                         if (match.start === groupMatch.start && match.end === groupMatch.end) {
#                             isInGroup = true;
#                             break;
#                         }
#                     }
                    
#                     if (isInGroup) break;
#                 }
                
#                 if (!isInGroup) {
#                     middleBrackets.push(match);
#                 }
#             }
            
#             // If we have middle brackets between groups
#             if (middleBrackets.length > 0) {
#                 // Create variant with first group content + middle brackets + last group content
#                 const complexVariantParts = [];
#                 let baseTextAdded = false;
                
#                 // Add first group content
#                 complexVariantParts.push(bracketGroups[0].matches[0].content);
                
#                 // FIXED: Use a new RegExp instance for replacement
#                 const baseText = cleanedTitle.replace(new RegExp(bracketPatternSource, 'g'), '');
#                 complexVariantParts.push(baseText);
#                 baseTextAdded = true;
                
#                 // Add content from middle brackets
#                 for (const middleMatch of middleBrackets) {
#                     complexVariantParts.push(middleMatch.content);
#                 }
                
#                 // Add last group content (first bracket only)
#                 complexVariantParts.push(bracketGroups[bracketGroups.length-1].matches[0].content);
                
#                 const complexVariant = removeArticles(normalizeWhitespace(complexVariantParts.join(' ')));
                
#                 if (complexVariant && 
#                     !variants.has(complexVariant) && 
#                     complexVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                     variants.add(complexVariant);
#                 }
                
#                 // Also create variant with base text + last group content
#                 if (!baseTextAdded) {
#                     const baseVariantParts = [
#                         baseText, 
#                         bracketGroups[bracketGroups.length-1].matches[0].content
#                     ];
                    
#                     const baseVariant = removeArticles(normalizeWhitespace(baseVariantParts.join(' ')));
                    
#                     if (baseVariant && 
#                         !variants.has(baseVariant) && 
#                         baseVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                         variants.add(baseVariant);
#                     }
#                 }
#             }
#         }
        
#         return Array.from(variants).sort();
#     $$;           
#     """).collect()
    
#     return


# def process_batches_sql(session, table_name):
#     """Process title variants using SQL and the registered JavaScript UDFs"""
#     print("Processing title variants using SQL...")
    
#     # Create a temporary table to store results
#     session.sql(f"""
#     CREATE OR REPLACE TEMPORARY TABLE EDW_APPS.MATCHING.TEMP_TITLE_VARIANTS AS
#     SELECT
#         t.*,
#         t.TITLE as ORIGINAL_TITLE,
#         NULL as TITLE_VARIANT,
#         FALSE as IS_VARIANT
#     FROM {table_name} t
#     """).collect()
    
#     # Process variants using the JavaScript UDF and insert into the temporary table
#     session.sql(f"""
#     INSERT INTO EDW_APPS.MATCHING.TEMP_TITLE_VARIANTS
#     SELECT 
#         t.*,
#         t.TITLE as ORIGINAL_TITLE,
#         v.value::STRING as TITLE_VARIANT,
#         TRUE as IS_VARIANT
#     FROM {table_name} t,
#     TABLE(FLATTEN(EDW_APPS.MATCHING.GENERATE_TITLE_VARIANTS_JS(t.TITLE))) v
#     WHERE v.value IS NOT NULL
#     """).collect()
    
#     # Update original rows to include their TITLE as TITLE_VARIANT if they have no variants
#     session.sql("""
#     UPDATE EDW_APPS.MATCHING.TEMP_TITLE_VARIANTS orig
#     SET TITLE_VARIANT = TITLE
#     WHERE IS_VARIANT = FALSE
#     AND NOT EXISTS (
#         SELECT 1 FROM EDW_APPS.MATCHING.TEMP_TITLE_VARIANTS var
#         WHERE var.ORIGINAL_TITLE = orig.TITLE
#         AND var.IS_VARIANT = TRUE
#     )
#     """).collect()
    
#     # Create the final table
#     session.sql("""
#     CREATE OR REPLACE TABLE EDW_APPS.MATCHING.ADC_WORKS_TITLE_VARIANTS AS
#     SELECT * FROM EDW_APPS.MATCHING.TEMP_TITLE_VARIANTS
#     //WHERE TITLE_VARIANT IS NOT NULL
#     """).collect()
    
#     stats = session.sql("""
#     SELECT
#         SUM(CASE WHEN IS_VARIANT = FALSE THEN 1 ELSE 0 END) as ORIGINAL_COUNT,
#         SUM(CASE WHEN IS_VARIANT = TRUE THEN 1 ELSE 0 END) as VARIANT_COUNT,
#         COUNT(*) as TOTAL_COUNT
#     FROM EDW_APPS.MATCHING.ADC_WORKS_TITLE_VARIANTS
#     """).collect()
    
#     print(f"OAriginal tracks: {stats[0]['ORIGINAL_COUNT']}")
#     print(f"Generated variants: {stats[0]['VARIANT_COUNT']}")
#     print(f"Total rows in output: {stats[0]['TOTAL_COUNT']}")
    
#     return


# def main(session):
#     # Get the input data from ADCWORKS
#     print("Reading data from ADCWORKS...")
#     table_name = "EDW_APPS.MATCHING.ADCWORKS"
    
#     # Register JavaScript UDFs for title variant generation
#     register_udfs(session)
    
#     # Process data in SQL batches
#     process_batches_sql(session, table_name)

    
#     print("\nTitle variant processing complete!")
#     return session.table("EDW_APPS.MATCHING.ADC_WORKS_TITLE_VARIANTS")

# # Run the main function
# result_df = main(session)

In [ ]:
# #MZK_TRACKS_TITLE_VARIANT_GENERATION

# import re
# import snowflake.snowpark.functions as F
# from snowflake.snowpark.functions import col, lit, regexp_replace, when

# # Register JavaScript UDFs for title variant generation
# def register_udfs(session):
#     print("Registering JavaScript UDFs for title variant generation...")
    
#     # Register the JavaScript UDFs
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.IS_ARTICLE(word STRING)
#     RETURNS BOOLEAN
#     LANGUAGE JAVASCRIPT
#     AS $$
#         const articles = ['A', 'AN', 'THE'];
#         return articles.includes(WORD.toUpperCase());
#     $$;
#     """).collect()
    
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.REMOVE_ARTICLES(title STRING)
#     RETURNS STRING
#     LANGUAGE JAVASCRIPT
#     AS $$
#         if (!TITLE) return '';
        
#         const articles = ['A', 'AN', 'THE'];
#         const words = TITLE.trim().split(/\s+/);
        
#         // Remove leading article
#         if (words.length > 0 && articles.includes(words[0].toUpperCase())) {
#             words.shift();
#         }
        
#         // Remove trailing article
#         if (words.length > 0 && articles.includes(words[words.length - 1].toUpperCase())) {
#             words.pop();
#         }
        
#         return words.join(' ');
#     $$;
#     """).collect()
    
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.NORMALIZE_WHITESPACE(title STRING)
#     RETURNS STRING
#     LANGUAGE JAVASCRIPT
#     AS $$
#         if (!TITLE) return '';
#         return TITLE.trim().replace(/\s+/g, ' ');
#     $$;
#     """).collect()
    
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.CLEAN_UNMATCHED_BRACKETS(title STRING)
#     RETURNS STRING
#     LANGUAGE JAVASCRIPT
#     AS $$
#         if (!TITLE) return '';
        
#         const stack = [];
#         const bracketPairs = {'(': ')', '[': ']', '{': '}'};
#         const reversePairs = {')': '(', ']': '[', '}': '{'};
#         const skipIndices = new Set();
        
#         // First pass - identify unmatched brackets
#         for (let i = 0; i < TITLE.length; i++) {
#             const char = TITLE[i];
            
#             if (bracketPairs[char]) {  // Opening bracket
#                 stack.push([char, i]);
#             } else if (reversePairs[char]) {  // Closing bracket
#                 if (stack.length && stack[stack.length - 1][0] === reversePairs[char]) {
#                     stack.pop();  // Matched pair
#                 } else {
#                     // Unmatched closing bracket - mark for removal
#                     skipIndices.add(i);
#                 }
#             }
#         }
        
#         // Any remaining opening brackets are unmatched
#         for (const [_, idx] of stack) {
#             skipIndices.add(idx);
#         }
        
#         // Build cleaned title without unmatched brackets
#         let cleanedTitle = "";
#         for (let i = 0; i < TITLE.length; i++) {
#             if (!skipIndices.has(i)) {
#                 cleanedTitle += TITLE[i];
#             }
#         }
        
#         return cleanedTitle.trim().replace(/\s+/g, ' ');
#     $$;
#     """).collect()
    
#     # Complex variant generation function that handles NULL titles
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.GENERATE_TITLE_VARIANTS_JS(title STRING)
#     RETURNS VARIANT
#     LANGUAGE JAVASCRIPT
#     AS $$
#         // Handle NULL titles
#         if (TITLE === null || TITLE === undefined) {
#             return [];
#         }
        
#         const originalTitle = TITLE.trim();
        
#         // Helper functions
#         function isArticle(word) {
#             const articles = ['A', 'AN', 'THE'];
#             return articles.includes(word.toUpperCase());
#         }
        
#         function removeArticles(title) {
#             if (!title) return '';
#             const words = title.trim().split(/\s+/);
            
#             if (words.length > 0 && isArticle(words[0])) {
#                 words.shift();
#             }
            
#             if (words.length > 0 && isArticle(words[words.length - 1])) {
#                 words.pop();
#             }
            
#             return words.join(' ');
#         }
        
#         function normalizeWhitespace(title) {
#             if (!title) return '';
#             return title.trim().replace(/\s+/g, ' ');
#         }
        
#         function cleanUnmatchedBrackets(title) {
#             if (!title) return '';
            
#             const stack = [];
#             const bracketPairs = {'(': ')', '[': ']', '{': '}'};
#             const reversePairs = {')': '(', ']': '[', '}': '{'};
#             const skipIndices = new Set();
            
#             // First pass - identify unmatched brackets
#             for (let i = 0; i < title.length; i++) {
#                 const char = title[i];
                
#                 if (bracketPairs[char]) {  // Opening bracket
#                     stack.push([char, i]);
#                 } else if (reversePairs[char]) {  // Closing bracket
#                     if (stack.length && stack[stack.length - 1][0] === reversePairs[char]) {
#                         stack.pop();  // Matched pair
#                     } else {
#                         // Unmatched closing bracket - mark for removal
#                         skipIndices.add(i);
#                     }
#                 }
#             }
            
#             // Any remaining opening brackets are unmatched
#             for (const [_, idx] of stack) {
#                 skipIndices.add(idx);
#             }
            
#             // Build cleaned title without unmatched brackets
#             let cleanedTitle = "";
#             for (let i = 0; i < title.length; i++) {
#                 if (!skipIndices.has(i)) {
#                     cleanedTitle += title[i];
#                 }
#             }
            
#             return normalizeWhitespace(cleanedTitle);
#         }
        
#         // Main variant generation logic
#         const cleanedTitle = cleanUnmatchedBrackets(originalTitle);
#         const variants = new Set();
        
#         // FIXED: Store only the regex pattern source, not the regex object itself
#         const bracketPatternSource = /([\[\(\{])([^\[\]\(\)\{\}]*)([\]\)\}])/g.source;
        
#         // Find all matched brackets in the cleaned title
#         const bracketMatches = [];
#         let match;
#         // FIXED: Always create a new RegExp instance when using the pattern
#         const patternForMatching = new RegExp(bracketPatternSource, 'g');
#         while ((match = patternForMatching.exec(cleanedTitle)) !== null) {
#             bracketMatches.push({
#                 full: match[0],
#                 open: match[1],
#                 content: match[2],
#                 close: match[3],
#                 start: match.index,
#                 end: match.index + match[0].length
#             });
#         }
        
#         // If no brackets, check if original had any
#         if (bracketMatches.length === 0) {
#             if (/[\[\]\(\)\{\}]/.test(originalTitle)) {
#                 const cleanTitle = removeArticles(cleanedTitle);
#                 if (cleanTitle) {
#                     variants.add(cleanTitle);
#                 }
#             }
#             return Array.from(variants).sort();
#         }
        
#         // FIXED: Create a new RegExp instance for each replacement
#         const noBrackets = cleanedTitle.replace(new RegExp(bracketPatternSource, 'g'), '');
#         const normalizedNoBrackets = normalizeWhitespace(noBrackets);
#         const noArticlesNoBrackets = removeArticles(normalizedNoBrackets);
        
#         if (noArticlesNoBrackets && 
#             noArticlesNoBrackets.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#             variants.add(noArticlesNoBrackets);
#         }
        
#         // Group adjacent brackets
#         const bracketGroups = [];
#         if (bracketMatches.length > 0) {
#             let currentGroup = [bracketMatches[0]];
            
#             for (let i = 1; i < bracketMatches.length; i++) {
#                 const prevEnd = bracketMatches[i-1].end;
#                 const currentStart = bracketMatches[i].start;
                
#                 // Check if there's text between brackets
#                 if (cleanedTitle.substring(prevEnd, currentStart).trim()) {
#                     // Non-adjacent brackets, start a new group
#                     bracketGroups.push({
#                         start: currentGroup[0].start,
#                         end: currentGroup[currentGroup.length-1].end,
#                         matches: [...currentGroup]
#                     });
#                     currentGroup = [bracketMatches[i]];
#                 } else {
#                     // Adjacent brackets, add to current group
#                     currentGroup.push(bracketMatches[i]);
#                 }
#             }
            
#             // Add the last group
#             bracketGroups.push({
#                 start: currentGroup[0].start,
#                 end: currentGroup[currentGroup.length-1].end,
#                 matches: [...currentGroup]
#             });
#         }
        
#         // Split the title into segments: text segments and bracket groups
#         const segments = [];
#         let lastEnd = 0;
        
#         for (const group of bracketGroups) {
#             // Add text before this group
#             if (group.start > lastEnd) {
#                 segments.push({
#                     type: 'text',
#                     content: cleanedTitle.substring(lastEnd, group.start)
#                 });
#             }
            
#             // Add this bracket group
#             segments.push({
#                 type: 'group',
#                 content: group.matches
#             });
            
#             // Update lastEnd
#             lastEnd = group.end;
#         }
        
#         // Add any remaining text after the last bracket group
#         if (lastEnd < cleanedTitle.length) {
#             segments.push({
#                 type: 'text',
#                 content: cleanedTitle.substring(lastEnd)
#             });
#         }
        
#         // Variant 1: with first bracket of first group only
#         const firstVariantParts = [];
        
#         for (const segment of segments) {
#             if (segment.type === 'text') {
#                 firstVariantParts.push(segment.content);
#             } else if (segment.type === 'group') {
#                 // Add only first bracket content from first group
#                 firstVariantParts.push(segment.content[0].content);
#             }
#         }
        
#         const firstVariant = removeArticles(normalizeWhitespace(firstVariantParts.join(' ')));
        
#         if (firstVariant && 
#             !variants.has(firstVariant) && 
#             firstVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#             variants.add(firstVariant);
#         }
        
#         // Variant 2: If multiple groups, create variant with first bracket of first group
#         // and first bracket of last group
#         if (bracketGroups.length > 1) {
#             const secondVariantParts = [];
#             let firstGroupUsed = false;
#             let lastGroupUsed = false;
            
#             for (const segment of segments) {
#                 if (segment.type === 'text') {
#                     secondVariantParts.push(segment.content);
#                 } else if (segment.type === 'group') {
#                     // Check if this is first group
#                     if (!firstGroupUsed && segment.content[0].start === bracketGroups[0].matches[0].start) {
#                         secondVariantParts.push(segment.content[0].content);
#                         firstGroupUsed = true;
#                     } 
#                     // Check if this is last group
#                     else if (!lastGroupUsed && 
#                             segment.content[0].start === bracketGroups[bracketGroups.length-1].matches[0].start &&
#                             bracketGroups[0].start !== bracketGroups[bracketGroups.length-1].start) {
#                         secondVariantParts.push(segment.content[0].content);
#                         lastGroupUsed = true;
#                     }
#                 }
#             }
            
#             const secondVariant = removeArticles(normalizeWhitespace(secondVariantParts.join(' ')));
            
#             if (secondVariant && 
#                 !variants.has(secondVariant) && 
#                 secondVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                 variants.add(secondVariant);
#             }
#         }
        
#         // Variant 3: If multiple groups, create variant with just text and first bracket of last group
#         if (bracketGroups.length > 1) {
#             const thirdVariantParts = [];
#             let lastGroupUsed = false;
            
#             for (const segment of segments) {
#                 if (segment.type === 'text') {
#                     thirdVariantParts.push(segment.content);
#                 } else if (segment.type === 'group' && !lastGroupUsed && 
#                         segment.content[0].start === bracketGroups[bracketGroups.length-1].matches[0].start) {
#                     thirdVariantParts.push(segment.content[0].content);
#                     lastGroupUsed = true;
#                 }
#             }
            
#             const thirdVariant = removeArticles(normalizeWhitespace(thirdVariantParts.join(' ')));
            
#             if (thirdVariant && 
#                 !variants.has(thirdVariant) && 
#                 thirdVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                 variants.add(thirdVariant);
#             }
#         }
        
#         // Variant 4: Handle titles that start with a bracket
#         if (bracketMatches.length > 0 && bracketMatches[0].start === 0) {
#             // Get just the content of the first bracket
#             const bracketContent = normalizeWhitespace(bracketMatches[0].content);
#             let restOfTitle = cleanedTitle.substring(bracketMatches[0].end);
            
#             // FIXED: Use a new RegExp instance for replacement
#             restOfTitle = restOfTitle.replace(new RegExp(bracketPatternSource, 'g'), '');
#             restOfTitle = normalizeWhitespace(restOfTitle);
            
#             // Create variant with just bracket content + rest of title
#             const combined = removeArticles(normalizeWhitespace(`${bracketContent} ${restOfTitle}`));
            
#             if (combined && 
#                 !variants.has(combined) && 
#                 combined.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                 variants.add(combined);
#             }
            
#             // Also create a variant with just the bracket content if it's followed by more brackets
#             if (bracketMatches.length > 1 && bracketMatches[1].start === bracketMatches[0].end) {
#                 const justContent = removeArticles(bracketContent);
                
#                 if (justContent && 
#                     !variants.has(justContent) && 
#                     justContent.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                     variants.add(justContent);
#                 }
#             }
#         }
        
#         // Complex case handling for titles like "(NEW WORLD) (HI) HELLO (SMILE TIME) WORLD (TIME) (XXXX)"
#         if (bracketGroups.length >= 2) {
#             // Find brackets that aren't in any group (middle brackets)
#             const middleBrackets = [];
            
#             for (const match of bracketMatches) {
#                 let isInGroup = false;
                
#                 for (const group of bracketGroups) {
#                     for (const groupMatch of group.matches) {
#                         if (match.start === groupMatch.start && match.end === groupMatch.end) {
#                             isInGroup = true;
#                             break;
#                         }
#                     }
                    
#                     if (isInGroup) break;
#                 }
                
#                 if (!isInGroup) {
#                     middleBrackets.push(match);
#                 }
#             }
            
#             // If we have middle brackets between groups
#             if (middleBrackets.length > 0) {
#                 // Create variant with first group content + middle brackets + last group content
#                 const complexVariantParts = [];
#                 let baseTextAdded = false;
                
#                 // Add first group content
#                 complexVariantParts.push(bracketGroups[0].matches[0].content);
                
#                 // FIXED: Use a new RegExp instance for replacement
#                 const baseText = cleanedTitle.replace(new RegExp(bracketPatternSource, 'g'), '');
#                 complexVariantParts.push(baseText);
#                 baseTextAdded = true;
                
#                 // Add content from middle brackets
#                 for (const middleMatch of middleBrackets) {
#                     complexVariantParts.push(middleMatch.content);
#                 }
                
#                 // Add last group content (first bracket only)
#                 complexVariantParts.push(bracketGroups[bracketGroups.length-1].matches[0].content);
                
#                 const complexVariant = removeArticles(normalizeWhitespace(complexVariantParts.join(' ')));
                
#                 if (complexVariant && 
#                     !variants.has(complexVariant) && 
#                     complexVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                     variants.add(complexVariant);
#                 }
                
#                 // Also create variant with base text + last group content
#                 if (!baseTextAdded) {
#                     const baseVariantParts = [
#                         baseText, 
#                         bracketGroups[bracketGroups.length-1].matches[0].content
#                     ];
                    
#                     const baseVariant = removeArticles(normalizeWhitespace(baseVariantParts.join(' ')));
                    
#                     if (baseVariant && 
#                         !variants.has(baseVariant) && 
#                         baseVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                         variants.add(baseVariant);
#                     }
#                 }
#             }
#         }
        
#         return Array.from(variants).sort();
#     $$;           
#     """).collect()
    
#     return


# def process_batches_sql(session, table_name):
#     """Process title variants using SQL and the registered JavaScript UDFs"""
#     print("Processing title variants using SQL...")
    
#     # Create a temporary table to store results
#     session.sql(f"""
#     CREATE OR REPLACE TEMPORARY TABLE EDW_APPS.MATCHING.TEMP_MZK_TITLE_VARIANTS AS
#     SELECT
#         t.*,
#         t.TITLE as ORIGINAL_TITLE,
#         NULL as TITLE_VARIANT,
#         FALSE as IS_VARIANT
#     FROM {table_name} t
#     """).collect()
    
#     # Process variants using the JavaScript UDF and insert into the temporary table
#     session.sql(f"""
#     INSERT INTO EDW_APPS.MATCHING.TEMP_MZK_TITLE_VARIANTS
#     SELECT 
#         t.*,
#         t.TITLE as ORIGINAL_TITLE,
#         v.value::STRING as TITLE_VARIANT,
#         TRUE as IS_VARIANT
#     FROM {table_name} t,
#     TABLE(FLATTEN(EDW_APPS.MATCHING.GENERATE_TITLE_VARIANTS_JS(t.TITLE))) v
#     WHERE v.value IS NOT NULL
#     """).collect()
    
#     # Update original rows to include their TITLE as TITLE_VARIANT if they have no variants
#     session.sql("""
#     UPDATE EDW_APPS.MATCHING.TEMP_MZK_TITLE_VARIANTS orig
#     SET TITLE_VARIANT = TITLE
#     WHERE IS_VARIANT = FALSE
#     AND NOT EXISTS (
#         SELECT 1 FROM EDW_APPS.MATCHING.TEMP_MZK_TITLE_VARIANTS var
#         WHERE var.ORIGINAL_TITLE = orig.TITLE
#         AND var.IS_VARIANT = TRUE
#     )
#     """).collect()
    
#     # Create the final table
#     session.sql("""
#     CREATE OR REPLACE TABLE EDW_APPS.MATCHING.MZK_TRACKS_TITLE_VARIANTS AS
#     SELECT * FROM EDW_APPS.MATCHING.TEMP_MZK_TITLE_VARIANTS
#     //WHERE TITLE_VARIANT IS NOT NULL
#     """).collect()
    
#     # Get statistics
#     stats = session.sql("""
#     SELECT
#         SUM(CASE WHEN IS_VARIANT = FALSE THEN 1 ELSE 0 END) as ORIGINAL_COUNT,
#         SUM(CASE WHEN IS_VARIANT = TRUE THEN 1 ELSE 0 END) as VARIANT_COUNT,
#         COUNT(*) as TOTAL_COUNT
#     FROM EDW_APPS.MATCHING.MZK_TRACKS_TITLE_VARIANTS
#     """).collect()
    
#     print(f"Original tracks: {stats[0]['ORIGINAL_COUNT']}")
#     print(f"Generated variants: {stats[0]['VARIANT_COUNT']}")
#     print(f"Total rows in output: {stats[0]['TOTAL_COUNT']}")
    
#     return


# def main(session):
#     # Get the input data from MAZOOKA TRACKS
#     print("Reading data from MAZOOKA...")
#     table_name = "EDW_APPS.MATCHING.TRACKS"
    
#     print("\nSample data (5 rows):")
#     session.table(table_name).limit(5).show()
    
    
#     # Register JavaScript UDFs for title variant generation
#     register_udfs(session)
    
#     # Process data in SQL batches
#     process_batches_sql(session, table_name)
    
#     return session.table("EDW_APPS.MATCHING.MZK_TRACKS_TITLE_VARIANTS")

# # Run the main function
# result_df = main(session)

In [ ]:
# # ADC_MZK_TITLE_VARIANTS_ARTICLES_CLEAN

# import time
# import snowflake.snowpark as snowpark


# # Define table names
# apra_table = "ADC_WORKS_TITLE_VARIANTS"
# mzk_table = "MZK_TRACKS_TITLE_VARIANTS"
# apra_output_table = "ADC_WORKS_TITLE_VARIANTS_CLEAN"
# mzk_output_table = "MZK_TRACKS_TITLE_VARIANTS_CLEAN"

# print("Starting title cleansing process...")
# start_time = time.time()

# # Create JavaScript function for title cleansing
# js_cleanse_title = """
# CREATE OR REPLACE FUNCTION JS_CLEANSE_TITLE(title VARCHAR)
# RETURNS VARCHAR
# LANGUAGE JAVASCRIPT
# AS
# $$
#   if (TITLE === null || TITLE === undefined) {
#     return "";
#   }
  
#   // Convert to uppercase
#   let result = TITLE.toUpperCase();
  
#   // Remove articles from beginning
#   result = result.replace(/^(A|AN|THE)\\s+/, '');
  
#   // Remove punctuation
#   result = result.replace(/[.,\\/#!$%^&*;:{}=\\-_`~()]/g, '');
  
#   // Remove extra whitespace
#   result = result.replace(/\\s+/g, ' ').trim();
  
#   return result;
# $$;
# """
# session.sql(js_cleanse_title).collect()
# print("Created JavaScript cleansing function")

# # Process ADC table with SQL
# print(f"Cleansing ADC titles and creating {apra_output_table}...")
# adc_cleanse_sql = f"""
# CREATE OR REPLACE TABLE {apra_output_table} AS
# SELECT 
#     APRA_WORK_ID,
#     ORIGINAL_TITLE,
#     JS_CLEANSE_TITLE(TITLE_VARIANT) AS APRA_CLEANED_TITLE,
#     ISWC AS APRA_ISWC,
#     CD_TYPE,
#     YN_PERF_OWNERSHIP
# FROM {apra_table}
# """
# session.sql(adc_cleanse_sql).collect()

# # Process MZK table with SQL
# print(f"Cleansing MZK titles and creating {mzk_output_table}...")
# mzk_cleanse_sql = f"""
# CREATE OR REPLACE TABLE {mzk_output_table} AS
# SELECT 
#     TRACK_ID AS MUZOOKA_TRACK_ID,
#     ORIGINAL_TITLE,
#     JS_CLEANSE_TITLE(TITLE_VARIANT) AS MUZOOKA_CLEANED_TITLE,
#     ISWC AS MUZOOKA_ISWC
# FROM {mzk_table}
# """
# session.sql(mzk_cleanse_sql).collect()

# # Count the number of rows in each cleansed table
# adc_count = session.sql(f"SELECT COUNT(*) AS COUNT FROM {apra_output_table}").collect()[0]["COUNT"]
# mzk_count = session.sql(f"SELECT COUNT(*) AS COUNT FROM {mzk_output_table}").collect()[0]["COUNT"]

# elapsed = time.time() - start_time
# print(f"Cleansing completed in {elapsed:.2f} seconds")
# print(f"Processed {adc_count} ADC records and {mzk_count} MZK records")

# print("Title cleansing completed successfully!")

In [ ]:


-- --EXACT TITLE MATCHING
-- CREATE OR REPLACE TABLE adc_works_exact_match_title_variants AS (
--   WITH cte1 AS (
--     SELECT 
--       apra_work_id,
--       ORIGINAL_TITLE AS APRA_ORIGINAL_TITLE,
--       apra_cleaned_title,
--       UPPER(REGEXP_REPLACE(APRA_ISWC, '[^A-Za-z0-9]', '')) AS apra_iswc,
--       CD_TYPE,
--       YN_PERF_OWNERSHIP
--     FROM ADC_WORKS_TITLE_VARIANTS_CLEAN 
--     WHERE apra_cleaned_title IS NOT NULL AND apra_cleaned_title <> ''
--   ),
--   cte2 AS (
--     SELECT 
--       ORIGINAL_TITLE AS MUZOOKA_ORIGINAL_TITLE,
--       UPPER(muzooka_track_id) AS muzooka_track_id,
--       muzooka_cleaned_title,
--       muzooka_iswc
--     FROM mzk_tracks_title_variants_clean
--     WHERE muzooka_cleaned_title IS NOT NULL AND muzooka_cleaned_title <> ''
--   )
--   SELECT DISTINCT
--     cte1.apra_work_id, 
--     cte1.APRA_ORIGINAL_TITLE,
--     cte1.apra_cleaned_title,
--     cte1.apra_iswc,
--     cte1.cd_type,
--     cte1.YN_PERF_OWNERSHIP,
--     cte2.muzooka_track_id,
--     cte2.MUZOOKA_ORIGINAL_TITLE,
--     cte2.muzooka_cleaned_title,
--     cte2.muzooka_iswc,
--     100 AS match_score,
--     CASE WHEN cte1.apra_iswc = cte2.muzooka_iswc THEN 'Y' ELSE 'N' END AS YN_ISWC_MATCH
--   FROM cte1 
--   JOIN cte2
--   ON UPPER(cte1.apra_cleaned_title) = UPPER(cte2.muzooka_cleaned_title)
--   -- where cte1.apra_work_id = 'GW00577031'
-- );





In [ ]:
-- -- OLD LOGIC NO LONGER NEEDED


-- --GET NON EXACT MATCH WORKS

-- CREATE OR REPLACE TABLE ADC_WORKS_NON_EXACT_MATCH_TITLE_VARIANTS AS
-- SELECT DISTINCT
--   a.apra_work_id,
--   a.ORIGINAL_TITLE AS APRA_ORIGINAL_TITLE,
--   UPPER(a.apra_cleaned_title) AS apra_cleaned_title,
--   UPPER(REGEXP_REPLACE(a.apra_iswc, '[^A-Za-z0-9]', '')) AS apra_iswc,
--   a.cd_type,
--   a.YN_PERF_OWNERSHIP
-- FROM adc_works_title_variants_clean a
-- WHERE a.apra_cleaned_title <> '' -- Filter empty titles first for better performance
-- AND NOT EXISTS (
--   SELECT 1 
--   FROM mzk_tracks_title_variants_clean m
--   WHERE UPPER(a.apra_cleaned_title) = UPPER(m.muzooka_cleaned_title)
-- );


In [ ]:
-- -- OLD LOGIC - FUZZY TITLE MATCHING

-- CREATE OR REPLACE TABLE FUZZY_TITLE_MATCHING AS (
--   WITH cte1 AS (
--     SELECT DISTINCT * FROM adc_works_non_exact_match_title_variants 
--     LIMIT 1000000
--   ),
--   cte2 AS (
--     SELECT * FROM mzk_tracks_title_variants_clean
--   )
--   SELECT DISTINCT
--     cte1.*, 
--     cte2.ORIGINAL_TITLE as MUZOOKA_ORIGINAL_TITLE,
--     UPPER(cte2.muzooka_track_id) AS muzooka_track_id, 
--     cte2.muzooka_cleaned_title, 
--     cte2.muzooka_iswc,
--     jarowinkler_similarity(cte1.apra_cleaned_title, cte2.muzooka_cleaned_title) AS match_score,
--     CASE 
--       WHEN cte1.apra_iswc = cte2.muzooka_iswc THEN 'Y' 
--       ELSE 'N' 
--     END AS yn_iswc_match
--   FROM cte1 
--   JOIN cte2
--     ON SUBSTRING(cte1.apra_cleaned_title, 1, 3) = SUBSTRING(cte2.muzooka_cleaned_title, 1, 3)
--   WHERE jarowinkler_similarity(cte1.apra_cleaned_title, cte2.muzooka_cleaned_title) >= 90
-- );




In [ ]:
-- -- CURRENT LOGIC - FUZZY TITLE MATCHING - ALL APRA vs NON-EXACT MATCHED MUZOOKA


CREATE OR REPLACE TABLE FUZZY_TITLE_MATCHING_32X AS (
  WITH cte1 AS (
    -- ALL APRA titles (including those that had exact matches)
    SELECT DISTINCT
      apra_work_id,
      ORIGINAL_TITLE AS APRA_ORIGINAL_TITLE,
      UPPER(apra_cleaned_title) AS apra_cleaned_title,
      UPPER(REGEXP_REPLACE(apra_iswc, '[^A-Za-z0-9]', '')) AS apra_iswc,
      cd_type,
      YN_PERF_OWNERSHIP
    FROM adc_works_title_variants_clean_32X
    WHERE apra_cleaned_title IS NOT NULL AND apra_cleaned_title <> ''
    LIMIT 1000000
  ),
  cte2 AS (
    -- Only Muzooka titles that did NOT get exact matches
    SELECT DISTINCT
      m.ORIGINAL_TITLE AS MUZOOKA_ORIGINAL_TITLE,
      UPPER(m.muzooka_track_id) AS muzooka_track_id,
      UPPER(m.muzooka_cleaned_title) AS muzooka_cleaned_title,
      m.muzooka_iswc
    FROM mzk_tracks_title_variants_clean_32X m
    WHERE m.muzooka_cleaned_title IS NOT NULL 
    AND m.muzooka_cleaned_title <> ''
    AND NOT EXISTS (
      SELECT 1 
      FROM adc_works_title_variants_clean a
      WHERE UPPER(a.apra_cleaned_title) = UPPER(m.muzooka_cleaned_title)
    )
  )
  SELECT DISTINCT
    cte1.apra_work_id,
    cte1.APRA_ORIGINAL_TITLE,
    cte1.apra_cleaned_title,
    cte1.apra_iswc,
    cte1.cd_type,
    cte1.YN_PERF_OWNERSHIP,
    cte2.muzooka_track_id,
    cte2.MUZOOKA_ORIGINAL_TITLE,
    cte2.muzooka_cleaned_title,
    cte2.muzooka_iswc,
    jarowinkler_similarity(cte1.apra_cleaned_title, cte2.muzooka_cleaned_title) AS match_score,
    CASE 
      WHEN cte1.apra_iswc = cte2.muzooka_iswc THEN 'Y' 
      ELSE 'N' 
    END AS yn_iswc_match
  FROM cte1 
  JOIN cte2
    ON SUBSTRING(cte1.apra_cleaned_title, 1, 3) = SUBSTRING(cte2.muzooka_cleaned_title, 1, 3)
  WHERE jarowinkler_similarity(cte1.apra_cleaned_title, cte2.muzooka_cleaned_title) >= 90
);

In [ ]:
TITLE MATCHING COMBINED RESULTS

CREATE OR REPLACE TABLE EDW_APPS.MATCHING.adc_works_columns_title_matches_combined  AS
SELECT 
    APRA_WORK_ID,
    APRA_CLEANED_TITLE,
    APRA_ORIGINAL_TITLE,
    APRA_ISWC,
    MUZOOKA_TRACK_ID,
    MUZOOKA_ORIGINAL_TITLE,
    MUZOOKA_CLEANED_TITLE,
    MUZOOKA_ISWC,
    MATCH_SCORE,
    YN_ISWC_MATCH,
    CD_TYPE,
    YN_PERF_OWNERSHIP
FROM 
    EDW_APPS.MATCHING.FUZZY_TITLE_MATCHING
UNION
SELECT 
    APRA_WORK_ID,
    APRA_CLEANED_TITLE,
    APRA_ORIGINAL_TITLE,
    APRA_ISWC,
    MUZOOKA_TRACK_ID,
    MUZOOKA_ORIGINAL_TITLE,
    MUZOOKA_CLEANED_TITLE,
    MUZOOKA_ISWC,
    MATCH_SCORE,
    YN_ISWC_MATCH,
    CD_TYPE,
    YN_PERF_OWNERSHIP
FROM 
    EDW_APPS.MATCHING.ADC_WORKS_EXACT_MATCH_TITLE_VARIANTS
WHERE 
    NOT EXISTS (
        SELECT 1 
        FROM EDW_APPS.MATCHING.FUZZY_TITLE_MATCHING f 
        WHERE f.APRA_WORK_ID = ADC_WORKS_EXACT_MATCH_TITLE_VARIANTS.APRA_WORK_ID
        AND f.MUZOOKA_TRACK_ID = ADC_WORKS_EXACT_MATCH_TITLE_VARIANTS.MUZOOKA_TRACK_ID
    );


In [ ]:

GENERATE RDC_WORKS_ID

CREATE OR REPLACE TABLE adc_works_columns_title_matches_combined AS
SELECT 
    DENSE_RANK() OVER (ORDER BY APRA_WORK_ID, MUZOOKA_TRACK_ID) AS rdc_works_id,
    *
FROM adc_works_columns_title_matches_combined;

